In [ ]:
from fastapi import FastAPI, Response
from fastapi.responses import HTMLResponse, StreamingResponse
import pickle
import matplotlib.pyplot as plt
import io
import os
from PIL import Image
from io import BytesIO

from apicall_input import main_api_call 
from data_extraction_v2 import main_extract_transform


In [ ]:

def normalize_user(row, mean_df, std_df):
    z = (row - mean_df) / std_df
    return z

def getMeanStd_user(data):
    # drop the date column while the normalisaition is going on
    data_no_date = data.drop(columns =['Date'], errors = 'ignore')
    mean = data_no_date.mean()
    std = data_no_date.std()
    std.replace(to_replace=0.0, value=0.01, inplace=True)
    return mean, std

def norm_user_data(df):
    """
    normalize the user data
    """
    # Get the mean + std dev
    user_test_means, user_test_std = getMeanStd_user(df.copy())
    # Normalize
    user_normalized = df.apply(lambda x: normalize_user(x, user_test_means,user_test_std), axis=1)
    user_normalized = user_normalized.drop(columns=[ 'Date'], errors='ignore')
    return user_normalized

In [ ]:
def runitall(email: str, password: str):
    """ 
    run all functions to get the image
            email: str : user email for api call
            password: str : user password for api call
        return: buffer_img : image buffer with the plot
    """
    # import model
    with open('..\models\mvp2best_logistic_model.pkl', 'rb') as file:
        model = pickle.load(file)   
    #pipeline steps    
    start_date, end_date, df_memory = main_api_call(email, password)
    df = main_extract_transform(start_date, end_date, df_memory)
    # add the normalisation step
    norm_df= norm_user_data(df)
    # make predictions
    df['injury probabilities'] = model.predict_proba(norm_df)[:, 1]
    # plot the probabilities over time with a rolling mean
    plt.figure(figsize=(10,5))
    plt.plot(df['Date'],df['injury probabilities'])# (remove bracket add following ) .rolling(window=5).mean())
    plt.xticks(df['Date'][::5], rotation=45, ha='right')
    # Save to buffer
    buffer_img = io.BytesIO()
    plt.savefig(buffer_img, format="png")
    buffer_img.seek(0)
    plt.close()  # free memory

    return buffer_img

In [ ]:
app = FastAPI(
    title = 'Run Injury Prediction API',
    description = 'Predicting injury risk based on GarminConnect running data'
)

In [ ]:
# Endpoint to handle form submission and show 'loading'
@app.get("/", response_class=HTMLResponse)
async def login_form():
    html_content = """
    <html>
        <head><title>Injury Risk Prediction</title></head>
        <body>
            <h2>Login to Generate Injury Risk Prediction</h2>
            <form action="/predict_and_visualize/" method="post" onsubmit="showLoading()">
                <label>Email:</label>
                <input type="text" name="email" required><br/>
                <label>Password:</label>
                <input type="password" name="password" required><br/>
                <button type="submit">Generate Prediction</button>
            </form>
            <p id="loading-message" style="display:none;">Processing... Please wait.</p>
            <script>
                function showLoading() {
                    document.getElementById('loading-message').style.display = 'block';
                }
            </script>
        </body>
    </html>
    """
    return HTMLResponse(content=html_content)

In [ ]:
@app.post("/predict_and_visualize/")
async def predict_and_visualize(email: str = Form(...), password: str = Form(...)):
    """
    perform spoof calculation and return an image
    """
    try:
        # run the full pipeline to get the image
        img =  runitall(email, password)
           
        return StreamingResponse(img, media_type="image/png")
    
    except Exception as e:
        return {"error": str(e)}, 500

    

In [29]:
runitall(email = '', password='')

Garmin Connect API - Activity Downloader
Login successful!
Activity data for 'indoor_cardio|21-08-2025|20130260263.csv' loaded into DataFrame.
Activity data for 'running|20-08-2025|20116740584.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|19-08-2025|20111594169.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|18-08-2025|20096824015.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|15-08-2025|20065742282.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|14-08-2025|20059835179.csv' loaded into DataFrame.
Activity data for 'treadmill_running|13-08-2025|20044942352.csv' loaded into DataFrame.
Activity data for 'indoor_cardio|12-08-2025|20034672816.csv' loaded into DataFrame.
Activity data for 'walking|10-08-2025|20012275661.csv' loaded into DataFrame.
Activity data for 'running|10-08-2025|20010223403.csv' loaded into DataFrame.
Activity data for 'treadmill_running|08-08-2025|19990936196.csv' loaded into DataFrame.
Activity data for 'indoor_c

INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


          Date  Day1 total km  Day1 km z3+  Day1 km z5  Day2-3 nr.sessions  \
0   2025-07-15           0.00         0.00         0.0                 1.0   
1   2025-07-16           6.40         5.40         0.0                 0.0   
2   2025-07-17           0.00         0.00         0.0                 1.0   
3   2025-07-18           0.00         0.00         0.0                 1.0   
4   2025-07-19           0.00         0.00         0.0                 0.0   
5   2025-07-20           8.18         6.18         0.0                 0.0   
6   2025-07-21           0.00         0.00         0.0                 1.0   
7   2025-07-22           0.00         0.00         0.0                 1.0   
8   2025-07-23           4.69         1.00         0.0                 0.0   
9   2025-07-24           0.00         0.00         0.0                 1.0   
10  2025-07-25           0.00         0.00         0.0                 1.0   
11  2025-07-26           0.00         0.00         0.0          